# Bronze to Silver: Data Cleaning and Transformation

This notebook reads the bronze-layer `brz_order_items` table from the `ecommerce` catalog and performs a comprehensive column-by-column data quality audit. Each column is inspected for data type issues, inconsistent values, nulls, and formatting problems. After cleaning and type-casting, the processed data is written to the silver-layer `slv_order_items` table.

**Pipeline context:** Bronze (raw) → **Silver (cleaned & typed)** → Gold (aggregated)

## Source and Target

| Source | Target | Rows |
| --- | --- | --- |
| `ecommerce.bronze.brz_order_items` | `ecommerce.silver.slv_order_items` | 183,378 |

## Column Transformation Summary

All 13 data columns arrive as `StringType` from the bronze layer. The table below summarizes the type casting and cleaning applied to each column:

| Column | Bronze Type | Silver Type | Cleaning Applied |
| --- | --- | --- | --- |
| `dt` | string | date | Cast to `date` using `to_date` |
| `order_ts` | string | timestamp | Cast to `timestamp` using `to_timestamp` |
| `customer_id` | string | string | Validated — all 16 chars, 88,495 distinct |
| `order_id` | string | string | Validated — all 6 chars, 104,756 distinct |
| `item_seq` | string | integer | Cast to `integer` |
| `product_id` | string | string | Validated — all 13 chars, 48,702 distinct |
| `quantity` | string | integer | Replaced `'Two'` with `'2'`, cast to `integer` |
| `unit_price_currency` | string | string | Validated — 7 valid currency codes |
| `unit_price` | string | float | Stripped `$` prefix, cast to `float` |
| `discount_pct` | string | float | Stripped `%` suffix, cast to `float` |
| `tax_amount` | string | float | Cast to `float` |
| `channel` | string | string | Replaced `web`→`Website`, `app`→`Application` |
| `coupon_code` | string | string | Filled 119,548 nulls with `'NA'` |

## Processing Steps

1. **Read** the bronze Delta table and assess dimensions (183,378 rows × 15 columns)
2. **Audit** — check nulls, duplicates, summary statistics, and schema
3. **Clean column-by-column** — cast types, fix formatting issues, handle nulls
4. **Write to Silver** — save as a managed Delta table with `mode("overwrite")`

In [0]:
# --- Section 1: Setup & Initial Data Loading ---
# Import PySpark SQL functions and type classes needed for data cleaning and type casting
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType, TimestampType, FloatType

In [0]:
# Define the Unity Catalog name used throughout the notebook
# for both reading (bronze) and writing (silver) tables
catalog_name = 'ecommerce'

In [0]:
# Read the raw bronze-layer order items table into a Spark DataFrame
# Source: ecommerce.bronze.brz_order_items
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_order_items")

# Preview the first 10 rows with full column values (no truncation)
df_bronze.show(10, truncate=False)

In [0]:
# --- Section 2: Initial Data Quality Assessment ---
# Check the dimensions of the dataset (number of rows and columns)
print(f"\nRow count: {df_bronze.count()}")
print(f"Column count: {len(df_bronze.columns)}\n")

In [0]:
# Inspect the schema — all columns arrive as strings from the bronze layer
# and will need to be cast to their proper types during cleaning
df_bronze.printSchema()

In [0]:
# Count nulls in each column to identify columns that need null-handling
df_bronze.select([
    F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
    for c in df_bronze.columns
]).show()

In [0]:
# Check for duplicate rows using the composite key (order_id + item_seq)
# Each order can have multiple line items, so this pair should be unique
duplicates = df_bronze.groupBy("order_id", 'item_seq').count().filter("count > 1")
duplicates.show()

In [0]:
# Generate summary statistics (count, mean, stddev, min, max) for all columns
# to get a quick overview of value ranges and distributions
df_bronze.describe().display()

## Data Quality Review & Action Items

From the above checks, we observed the following issues. These are the next action items — we will review again and add necessary action items as needed.

| # | Observation | Action Item |
|---|-------------|-------------|
| 1 | Columns may not have the correct data types assigned | Correctly assign the appropriate data types for each and every column |
| 2 | Date columns are not in the proper date format | Change the data type of date columns to the proper date format |
| 3 | Null count is high for the `coupon_code` column | Investigate and address the high null count in the `coupon_code` column |
| 4 | The `quantity` column contains mixed data types (both numbers and strings) | Convert all string values in the `quantity` column to integer type |
| 5 | The `discount_pct` column values contain a `%` suffix | Strip the `%` character from `discount_pct` values and cast the column to a float type |
| 6 | Some values in `unit_price` column contain a `$` prefix | Strip the `$` character from `unit_price` values and cast the column to a float type |
| 7 | Each column should be individually reviewed for any remaining data quality issues | Perform a comprehensive column-by-column data quality audit and address any issues found |

In [0]:
# --- Section 3: Column-by-Column Data Cleaning ---
# 3.1  Date & Timestamp Columns: dt, order_ts
# Cast dt from String to Date and order_ts from String to Timestamp

df_bronze = df_bronze \
                .withColumn(
                    "dt",
                    F.to_date("dt", "yyyy-MM-dd")
                ) \
                .withColumn(
                    "order_ts",
                    F.to_timestamp("order_ts", 'yyyy-MM-dd HH:mm:ss')
                )

# Verify the updated schema
df_bronze.printSchema()

In [0]:
# 3.2  ID Columns: customer_id, order_id, item_seq, product_id
# --- Column Check: customer_id ---
# Check distinct count and verify all customer_id values have consistent length

print(f"Distinct Value count in customer_id column: {df_bronze.select("customer_id").distinct().count()}\n")

max = 0
min = len(df_bronze.select("customer_id").first()["customer_id"])
for value in df_bronze.select("customer_id").collect():
    if len(value["customer_id"]) > max:
        max = len(value["customer_id"])
    elif len(value["customer_id"]) < min:
        min = len(value["customer_id"])

print(f"Minimum length of customer_id value: {min}")
print(f"Maximum length of customer_id value: {max}")

**customer_id** — looks good. The data type is String, which is correct for alphanumeric IDs. All values are consistently 16 characters long with 88,495 distinct values across 183,378 rows.

In [0]:
# --- Column Check: order_id ---
# Check distinct count and verify all order_id values have consistent length

print(f"Distinct Value count in order_id column: {df_bronze.select("order_id").distinct().count()}\n")

max = 0
min = len(df_bronze.select("order_id").first()["order_id"])
for value in df_bronze.select("order_id").collect():
    if len(value["order_id"]) > max:
        max = len(value["order_id"])
    elif len(value["order_id"]) < min:
        min = len(value["order_id"])

print(min)
print(max)

**order_id** — looks good. All values are consistently 6 characters long with 104,756 distinct values. We will keep the data type as String since no numeric operations will be performed on it.

In [0]:
# --- Column Check: item_seq ---
# Check distinct values, count, and length consistency for item_seq

df_bronze.select("item_seq").distinct().show()
print(f"\nDistinct Value count in item-seq column: {df_bronze.select("item_seq").distinct().count()}\n")

max = 0
min = len(df_bronze.select("item_seq").first()["item_seq"])
for value in df_bronze.select("item_seq").collect():
    if len(value["item_seq"]) > max:
        max = len(value["item_seq"])
    elif len(value["item_seq"]) < min:
        min = len(value["item_seq"])

print(min)
print(max)

**item_seq** — values are clean (integers 1–5 representing line item sequence within an order). We need to cast the data type from String to Integer.

In [0]:
# Cast item_seq from String to Integer
df_bronze = df_bronze.withColumn(
    "item_seq",
    F.col("item_seq").cast(IntegerType())
)

In [0]:
# --- Column Check: product_id ---
# Check distinct count and verify all product_id values have consistent length

# df_bronze.select("product_id").distinct().show()
print(f"\nDistinct Value count in product_id column: {df_bronze.select("product_id").distinct().count()}\n")

max = 0
min = len(df_bronze.select("product_id").first()["product_id"])
for value in df_bronze.select("product_id").collect():
    if len(value["product_id"]) > max:
        max = len(value["product_id"])
    elif len(value["product_id"]) < min:
        min = len(value["product_id"])

print(min)
print(max)

**product_id** — looks good. All values are consistently 13 characters long with 48,702 distinct products. The data type (String) is correct for alphanumeric IDs.

In [0]:
# --- Column Check: quantity ---
# Inspect distinct values to identify any non-numeric entries
df_bronze.select("quantity").distinct().show()

**quantity** — the value `2` is spelled out as `'Two'` in some rows. We will replace `'Two'` with `'2'` and then cast the column to Integer type.

In [0]:
# Replace the string 'Two' with numeric '2', then cast quantity to Integer
df_bronze = df_bronze.withColumn(
    "quantity",
    F.regexp_replace(F.col("quantity"), 'Two', '2')
).withColumn(
    "quantity",
    F.col("quantity").cast(IntegerType())
)

In [0]:
# Verify the updated schema after casting quantity to Integer
df_bronze.printSchema()

In [0]:
# --- Column Check: unit_price_currency ---
# Inspect distinct currency values
df_bronze.select("unit_price_currency").distinct().show()

**unit_price_currency** — looks good. All values are valid currency codes. The data type (String) is appropriate for this column.

In [0]:
# --- Column Check: unit_price ---
# Inspect distinct values to identify any formatting issues (e.g. '$' prefix)
df_bronze.select("unit_price").distinct().show()

**unit_price** — some values contain a `$` prefix. We will strip the `$` character and cast the column to Float type.

In [0]:
# Remove the '$' prefix from unit_price values and cast to Float
df_bronze = df_bronze.withColumn(
    "unit_price", F.regexp_replace(F.col("unit_price"), "\\$", "")
).withColumn("unit_price", F.col("unit_price").cast(FloatType()))

In [0]:
# Verify the updated schema after casting unit_price to Float
df_bronze.printSchema()

In [0]:
# --- Column Check: discount_pct ---
# Inspect values to identify formatting issues (e.g. '%' suffix)

# df_bronze.select("discount_pct").distinct().show()
print(f"Distinct discount_pct values count: {df_bronze.select("discount_pct").distinct().count()}\n")

df_bronze.select("discount_pct").show(20)


**discount_pct** — values contain a `%` suffix that needs to be removed before casting to Float.

In [0]:
# Remove the '%' suffix from discount_pct values and cast to Float
df_bronze = df_bronze.withColumn(
    "discount_pct",
    F.regexp_replace(F.col("discount_pct"), '%', '').cast(FloatType())
)

# Verify schema and preview cleaned values
df_bronze.printSchema()

df_bronze.select("discount_pct").show(10)

In [0]:
# Preview the cleaned data so far to visually verify transformations
df_bronze.show(20)

In [0]:
# Re-check null counts after cleaning to verify remaining nulls
df_bronze.select([
    F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
    for c in df_bronze.columns
]).show()

In [0]:
# --- Column Check: tax_amount ---
# Review summary statistics and distinct values
df_bronze.select("tax_amount").describe().show()

df_bronze.select("tax_amount").distinct().show()

**tax_amount** — all values are numeric and look clean. We just need to cast the data type from String to Float.

In [0]:
# Cast tax_amount from String to Float
df_bronze = df_bronze.withColumn(
    "tax_amount",
    F.col("tax_amount").cast(FloatType())
)

# Verify the updated schema
df_bronze.printSchema()

In [0]:
# --- Column Check: channel ---
# Inspect distinct channel values
df_bronze.select("channel").distinct().show()

**channel** — contains two distinct values: `web` and `app`. We will standardize these to `Website` and `Application` for better readability.

In [0]:
# Replace abbreviated channel names with full descriptive names
# 'web' -> 'Website', 'app' -> 'Application'
df_bronze = df_bronze.withColumn(
    "channel",
    F.regexp_replace(F.col("channel"), 'web', 'Website')
).withColumn(
    "channel",
    F.regexp_replace(F.col("channel"), 'app', 'Application')
)

In [0]:
# --- Column Check: coupon_code ---
# Check distinct values, count, and null frequency

# checking Distinct values and count
df_bronze.select("coupon_code").distinct().show(10)
print(f"Distinct count: {df_bronze.select("coupon_code").distinct().count()}")

# null check
df_bronze.select([
    F.sum(F.col("coupon_code").isNull().cast(IntegerType())).alias("coupon_code")
]).show()

**coupon_code** — has a significant number of nulls (many orders simply don't use coupons). We will replace null values with `'NA'` to indicate "not applicable".

In [0]:
# Replace null values with 'NA' in the coupon_code column
df_bronze = df_bronze.fillna("NA", subset=["coupon_code"])

# Verify that no nulls remain in coupon_code
df_bronze.select([
    F.sum(F.col("coupon_code").isNull().cast(IntegerType())).alias("coupon_code")
]).show()

# Write to Silver Layer

All columns have now been cleaned and type-cast. The processed DataFrame is ready to be written to the silver-layer table `slv_order_items`.

In [0]:
# Write the cleaned and type-cast DataFrame to the silver-layer Delta table
# Target: ecommerce.silver.slv_order_items
# Using overwrite mode with mergeSchema to allow schema evolution
df_bronze.write \
        .mode("overwrite") \
        .format("delta") \
        .option("mergeSchema", True) \
        .saveAsTable(f"{catalog_name}.silver.slv_order_items")

## Summary

All 13 data columns in the `brz_order_items` table have been audited, cleaned, and type-cast. The processed data has been written to the silver layer:

| Property | Value |
| --- | --- |
| Silver Table | `ecommerce.silver.slv_order_items` |
| Source | `ecommerce.bronze.brz_order_items` |
| Total Rows | 183,378 (unchanged — no rows dropped) |
| Total Columns | 15 (13 data + 2 metadata) |
| Write Mode | `overwrite` with `mergeSchema=True` |

### Key Data Quality Issues Found and Resolved

| Column | Issue | Resolution |
| --- | --- | --- |
| `dt` | Stored as string | Cast to `date` type |
| `order_ts` | Stored as string | Cast to `timestamp` type |
| `item_seq` | Stored as string | Cast to `integer` |
| `quantity` | Mixed types — value `'Two'` instead of `2` | Replaced `'Two'` with `'2'`, cast to `integer` |
| `unit_price` | `$` prefix on some values | Stripped `$`, cast to `float` |
| `discount_pct` | `%` suffix on all values | Stripped `%`, cast to `float` |
| `tax_amount` | Stored as string | Cast to `float` |
| `channel` | Abbreviated names (`web`, `app`) | Replaced with `Website` and `Application` |
| `coupon_code` | 119,548 nulls (65% of rows) | Filled with `'NA'` |

### Columns Validated (No Changes Needed)

* `customer_id` — all 16 chars, 88,495 distinct values
* `order_id` — all 6 chars, 104,756 distinct values
* `product_id` — all 13 chars, 48,702 distinct values
* `unit_price_currency` — 7 valid currency codes (GBP, INR, AED, USD, SGD, AUD, CAD)

No duplicate rows were found on the composite key (`order_id`, `item_seq`).